<a target="_blank" href="https://colab.research.google.com/drive/1H3ou3ooI26gFxVJr1kRN76K3rgawoXmf?usp=sharing">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Sichere Programmierung mit C/C++

C und C++ gehören zu den mächtigsten und einflussreichsten Programmiersprachen der Informatik. Sie ermöglichen direkten Zugriff auf Hardware, hochoptimierte Programme und feinste Kontrolle über Speicherverwaltung.

Mit dieser großen Freiheit kommt jedoch auch große Verantwortung:
C und C++ überlassen nahezu die komplette Speicherverwaltung dem Programmierer. Dadurch sind sie zwar extrem flexibel, aber es ist auch sehr einfach, schwere Fehler zu machen. Ein fehlendes `free()`, oder das Schreiben außerhalb eines Arrays kann sofort zu undefiniertem Verhalten, Programmabstürzen oder gravierenden Sicherheitslücken führen.

Dieses Notebook soll typische Formen unsicheren Speichermanagements in C/C++ demonstrieren, erklären und sichtbar machen. Dabei geht es nicht darum, die Sprachen schlechtzureden – sondern zu verstehen, wie leicht Fehler entstehen können und wie wichtig es ist, die zugrundeliegenden Konzepte wirklich zu beherrschen.

# Einleitung

In diesem Notebook untersuchen wir typische Fehler im Umgang mit Speicher in C/C++, darunter:


- Buffer Overflow (Stack und Heap)

- Pointers:

  - Use-After-Free

  - Double Free

  - Dangling Pointer

  - Pointer auf ein Element in `std::vector<>`

- Out-of-bounds Read/Write

- Unsichere Funktionen

## Buffer Overflow (Stack)

Ein Buffer Overflow tritt auf, wenn ein Array über seine festgelegten Grenzen hinaus beschrieben wird. Dies kann beispielsweise passieren, wenn ein String als Eingabe verarbeitet wird, der die Größe des zugewiesenen Puffers überschreitet und die Länge des Strings nicht überprüft wird.

Im folgenden Beispiel wird ein String aus den Kommandozeilenargumenten eingelesen und in einem Puffer gespeichert. Anschließend wird der Puffer mit printf() ausgegeben:

In [ ]:
%%writefile bufferoverflow.cpp

#include <string.h>
#include <stdio.h>

#define BUFFER_SIZE 8

int main(int argc, char* argv[]) {

  char buffer[BUFFER_SIZE];

  strcpy(buffer, argv[1]);

  printf("%s\n",buffer);

  return 0;

}

Hiermit führen wir das Programm mit dem Eingabestring *test* aus:

In [ ]:
!g++ bufferoverflow.cpp -o bufferoverflow
!./bufferoverflow test

Dies funktioniert wie erwartet, weil der Puffer groß genug ist, um den String zu speichern. Falls wir jedoch das Programm mit folgenden Eingabestring starten...:

In [ ]:
!./bufferoverflow dieserstringistvielzugrossfuerdenpuffer

… stürzt das Programm ab bzw. der Compiler erkennt `(*** stack smashing detected ***: terminated)`, dass versucht wurde, über die Grenzen des Puffers hinauszuschreiben. Dadurch können möglicherweise wichtige Daten in benachbarten Speicherbereichen überschrieben werden. In der Folge ist die Stabilität und Sicherheit des Programms nicht mehr gewährleistet.

Um einen Buffer Overflow zu vermeiden, muss unbedingt sichergestellt werden, dass niemals ein String, dessen Länge den verfügbaren Speicherplatz überschreitet, in den Puffer geschrieben wird. Ein Beispiel für eine sichere Implementierung:

In [ ]:
%%writefile bufferoverflowfixed.cpp

#include <string.h>
#include <stdio.h>

#define BUFFER_SIZE 8

int main(int argc, char* argv[]) {

  char buffer[BUFFER_SIZE];

  int length = strlen(argv[1]);

  if(length >= BUFFER_SIZE) {
    printf("String ist zu groß! Exiting...");
    return -1;
  }

  strcpy(buffer, argv[1]);
  printf("%s\n",buffer);

  return 0;

}

In [ ]:
!g++ bufferoverflowfixed.cpp -o bufferoverflowfixed
!./bufferoverflowfixed dieserstringistvielzugrossfuerdenpuffer

Nun wird die Funktion `strcpy()` nicht mehr ausgeführt, sondern das Programm terminiert sofort, womit ein Buffer-Overflow verhindert wird. Es ist entscheidend, dass Benutzer-Eingaben in C/C++ **grundsätzlich als potenziell gefährlich** betrachtet werden müssen.

## Buffer Overflow (Heap)

Auch im Heap kann ein Buffer Overflow auftreten:

In [ ]:
%%writefile heapoverflow.cpp

#include <stdlib.h>
#include <string.h>
#include <stdio.h>

int main() {

    char *p = (char*)malloc(5); // 5 Bytes großer Puffer
    strcpy(p, "dieserstringistvielzugrossfuerdenpuffer");  // überschreibt Grenzen des Heap-Blocks
    printf("%s\n", p);
    free(p);
    return 0;

}

In [ ]:
!g++ heapoverflow.cpp -o heapoverflow -Wno-stringop-overflow
!./heapoverflow

Auch hier erkennt die Runtime, dass durch den Buffer Overflow, benachbarte Speicherbereiche überschrieben worden sind und dadurch wird das Programm frühzeitig terminiert. Hier sollte genau der selbe Lösungsansatz wie beim Stack Overflow genutzt werden.

## Use-after-Free

Ein Use-after-Free tritt auf, wenn ein Pointer weiterverwendet wird, nachdem bereits `free()` auf ihn aufgerufen wurde. Dadurch zeigt der Pointer auf Speicher, der nicht mehr gültig ist, was zu undefiniertem Verhalten und potenziell schwerwiegenden Fehlern führen kann.

In [ ]:
%%writefile useafterfree.cpp

#include <cstdlib>
#include <iostream>

int main() {

  int* value = (int*)malloc(sizeof(int));

  *value = 5;

  free(value);

  std::cout << *value << std::endl;


  return 0;

}


In [ ]:
!g++ useafterfree.cpp -o useafterfree
!./useafterfree

In diesem Beispiel sieht man, dass nach dem `free()` nicht mehr zuverlässig der zuvor gespeicherte Wert 5 ausgegeben wird, sondern ein zufälliger bzw. undefinierter Wert erscheint. Das liegt daran, dass der Pointer nach `free()` ungültig ist und jeder Zugriff darauf zu undefiniertem Verhalten führt.

Eine sinnvolle Gegenmaßnahme besteht darin, den Pointer unmittelbar nach einem `free()`-Aufruf auf `nullptr` (bzw. `NULL`) zu setzen. So lässt sich vor jedem Zugriff einfach prüfen, ob der Pointer noch gültig ist. Dadurch werden versehentliche Nutzungen bereits freigegebener Speicherbereiche zuverlässig verhindert:

In [ ]:
%%writefile useafterfreefixed.cpp

#include <cstdlib>
#include <iostream>

void freeSafe(int** ptr) {
  free(*ptr);
  *ptr = nullptr;
}

int main() {

  int* value = (int*)malloc(sizeof(int));

  *value = 5;

  freeSafe(&value);

  if(value == nullptr)
    std::cout << "Trying to access nullptr! Exiting..." << std::endl;
  else
    std::cout << *value << std::endl;


  return 0;

}


In [ ]:
!g++ useafterfreefixed.cpp -o useafterfreefixed
!./useafterfreefixed

## Double Free

Ein Double Free tritt auf, wenn derselbe Speicherbereich mehr als einmal durch `free()` freigegeben wird. Nach dem ersten `free()` ist der Pointer bereits ungültig („dangling pointer“). Ein erneuter `free()`-Aufruf auf denselben Pointer führt zu undefiniertem Verhalten: das Programm kann crashen, Speicherstrukturen können beschädigt werden oder es kann scheinbar problemlos weiterlaufen.

In [ ]:
%%writefile doublefree.cpp

#include <cstdlib>
#include <iostream>

int main() {

  int* value = (int*)malloc(sizeof(int));

  free(value);
  free(value);

  std::cout << "Test" << std::endl;

  return 0;


}

In [ ]:
!g++ doublefree.cpp -o doublefree
!./doublefree

Nach einem Double Free lässt sich gut beobachten, dass das Programm in der Regel frühzeitig terminiert oder sich auf andere Weise kritisch verhält. Welche Auswirkungen genau auftreten, hängt vom System und dem Allocator ab. Möglich sind sofortige Fehlermeldungen, Abstürze oder sogar stille Speicher­korruption. Dieses Verhalten zeigt deutlich, wie gefährlich ein mehrfaches Freigeben desselben Speicherbereichs ist und warum solche Fehler unbedingt vermieden werden sollten.

## Dangling Pointer

Ein dangling pointer bezeichnet einen Zeiger, der auf Speicher zeigt, der nicht mehr gültig ist – etwa weil er bereits freigegeben mit `free()`, verschoben oder der entsprechende Gültigkeitsbereich verlassen wurde. Solche ungültigen Zeiger führen leicht zu schwer nachvollziehbaren Fehlern und können beim Zugriff zu undefiniertem Verhalten führen.

In [ ]:
%%writefile danglingpointer.cpp

#include <iostream>

int* createPtr() {

  int x = 123;
  return &x; // Zeigt auf Stack-Variable, die nach dem return zerstört wird

}


int main() {

  int* ptr = createPtr();

  *ptr = 5;

  std::cout << *ptr << std::endl; // Gibt nichts aus, da Programm crashed

  return 0;

}

In [ ]:
!g++ danglingpointer.cpp -o danglingpointer -Wno-return-local-addr
!./danglingpointer

Dieses Beispiel zeigt einen klassischen dangling pointer: Die Funktion gibt die Adresse einer lokalen Variable zurück, die nach Verlassen der Funktion nicht mehr existiert. Der Zeiger im `main()` zeigt somit auf ungültigen Speicher. Das Programmverhalten ist daher undefiniert. In diesem Fall stürtzt das Programm ab `Segmentation fault (core dumped)`. Solche Situationen machen deutlich, warum der Umgang mit Zeigern besondere Sorgfalt erfordert.

## Pointer auf ein Element in `std::vector<>`

Ein std::vector kann seine Elemente im Speicher verschieben, insbesondere wenn er wächst oder sich seine Kapazität ändert. Dadurch verlieren alle zuvor gespeicherten Pointer (und auch Referenzen und Iteratoren) auf seine Elemente ihre Gültigkeit. Das führt zu *dangling pointers* und damit zu undefiniertem Verhalten. Besonders tückisch ist, dass dieser Fehler nur bei bestimmten Größen oder Reservesituationen auftritt und daher oft schwer zu finden ist.

In [ ]:
%%writefile vectorpointer.cpp

#include <iostream>
#include <vector>

int main() {

    std::vector<int> v;
    v.push_back(10);
    v.push_back(20);

    int* ptr = &v[0];  // Pointer auf erstes Element

    // Durch push_back kann eine Reallokation stattfinden,
    // die den gesamten Speicher des Vektors verschiebt.
    v.push_back(30);

    // ptr zeigt jetzt möglicherweise ins Leere
    std::cout << *ptr << std::endl;  // Der Wert 10 sollte ausgegeben werden...

    return 0;
}

In [ ]:
!g++ vectorpointer.cpp -o vectorpointer
!./vectorpointer

Dieses Beispiel zeigt, wie schnell Pointer auf `std::vector`-Elemente ungültig werden können. Sobald der Vektor wächst und intern neuen Speicher allokiert, werden alle zuvor gehaltenen Adressen ungültig. Wer solche Pointer weiter nutzt, arbeitet unbemerkt mit *dangling pointers* und riskiert undefiniertes Verhalten. Von unerklärlichen Ausgaben bis hin zu Abstürzen. Deshalb sollte man in modernen C++-Programmen vermeiden, rohe Pointer auf Vektorelemente dauerhaft zu speichern und stattdessen Iteratoren oder stabile Datenstrukturen verwenden.

Um pointerbasierte Fehler zu vermeiden, werden in C++ sogennante *Smart Pointer* verwendet, dazu gibt es ein eigenes Notebook.

## Out-of-bounds Read/Write

Eine der häufigsten Ursachen für Buffer Overflows sind falsche Indexierungen bei Array-Operationen. Besonders in Schleifen, bei denen die Schleifenbedingungen oder Grenzen nicht korrekt gesetzt sind, kann es leicht dazu kommen, dass mehr Daten in einen Puffer geschrieben werden, als dieser tatsächlich aufnehmen kann. Dadurch werden angrenzende Speicherbereiche überschrieben, was zu unerwartetem Verhalten und Sicherheitslücken führen kann.

In [ ]:
%%writefile outofbounds.cpp

#include <iostream>

#define BUFFER_SIZE 8

int main() {

  int buffer[8];

  // Schleife geht bis i = 18, aber Puffer hat nur 8 Plätze
  for(size_t i = 0; i < BUFFER_SIZE + 10; i++) {
    buffer[i] = i;
  }

  return 0;


}

In [ ]:
!g++ outofbounds.cpp -o outofbounds
!./outofbounds

Um solche Out-of-Bounds Reads/Writes zu verhindern, nutzt man `std::array<T>` oder `std::vector<T>` statt Roharrays.

In [ ]:
%%writefile outofboundsfixed.cpp

#include <iostream>
#include <array>

#define BUFFER_SIZE 8

int main() {

  std::array<int,BUFFER_SIZE> buffer;

  // Schleife geht bis i = 18, aber Puffer hat nur 8 Plätze
  for(size_t i = 0; i < BUFFER_SIZE + 10; i++) {
    buffer.at(i) = i;
  }

  return 0;

}

In [ ]:
!g++ outofboundsfixed.cpp -o outofboundsfixed
!./outofboundsfixed

Die Memberfunktion `.at(index)` überprüft, ob der `index` noch im Pufferbereich ist. Falls nicht, wird eine Ausnahme ausgelöst. Dadurch werden Read/Write-Operationen nicht mehr ausgeführt.

## Unsichere Funktionen

Manche Funktionen sollten **unter keinen Umständen** in produktivem Code verwendet werden, da ihre Implementierung grundsätzlich unsicher ist und ein hohes Risiko für Speicherfehler oder Sicherheitslücken mit sich bringt. Eine unsichere Funktion haben wir schon im Buffer-Overflow-Beispiel kennengelernt, nähmlich `strcpy()`, da Kopieren ohne Längenüberprüfung stattfindet.
<br></br>
| Unsicher             | Sicher(er)                 |
| -------------------- | -------------------------- |
| `gets()`             | `fgets()`, `getline()`     |
| `strcpy()`           | `strlcpy()`, `strncpy()`   |
| `strcat()`           | `strlcat()`, `strncat()`   |
| `sprintf()`          | `snprintf()`               |
| `scanf("%s")`        | `scanf("%Ns")`, `fgets()`  |
| `strtok()`           | `strtok_r()`, `strtok_s()` |
| `bcopy()`, `bzero()` | `memcpy()`, `memset()`     |
| `realpath(NULL)`     | `realpath(path, buffer)`   |
| `system()`           |kein Userinput auf `system()` erlauben|

<br></br>
Beispiele:
```
Unsicher:
char buffer[10];
buffer = gets(); //gets() überprüft nicht die Länge des Eingabestring von der Konsole -> Buffer Overflow Gefahr
scanf("%s",buffer); //Genau wie gets()

Sicherer:
fgets(buffer, sizeof(buffer), stdin); //Größe des Puffers muss als Parameter übergeben werden
```


```
Unsicher:
char buffer[10];
int a = 5, b = 6;
buffer = sprintf(buffer, "%d plus %d is %d", a, b, a+b); //String ist zu groß für buffer, schreibt aber trotzdem alles rein

Sicherer:
buffer = snprintf(buffer, sizeof(buffer),"%d plus %d is %d", a, b, a+b); //String ist zu groß, wird auf Länge 10 geschnitten
```







## Übung:

Wir betrachten einen Code mit Fehlern, die wir oben kennengelernt haben. Ihre Aufgabe ist, alle Fehler zu erkennen und zu beheben:

In [ ]:
%%writefile uebung.cpp


#include <iostream>
#include <stdio.h>
#include <vector>

#define MAX 20

int main() {

  char name[MAX];

  printf("Gebe deinen Namen ein: ");
  scanf("%s", name);

  printf("Hallo %s \n", name);

  std::vector<int> numbers;

  for(size_t i = 0; i < 10;  i++) {
    numbers.push_back(i);
  }

  int* ptr = &numbers[3];
  *ptr = 33;


  for(size_t i = 0; i < 10;  i++) {
    numbers.push_back(i);
  }

  std::cout << "Wert von numbers[3]: " << *ptr << "\n";

  int* myArray = new int[10];

  for(size_t i = 0; i < 12; i++) {

    *myArray = i;

  }

  delete[] myArray;

  std::cout << "Wert von myArray[2]; " << myArray[2] << "\n";

  delete[] myArray;

  return 0;

}

In [ ]:
!g++ uebung.cpp -o uebung
!./uebung